In [1]:
import pandas as pd
import numpy as np
from math import radians, sin, cos, sqrt, atan2

# Load data
pois = pd.read_csv('Dataset.csv')
hexagons = pd.read_csv('Grid.csv')

# Convert coordinate columns to numeric, coercing errors to NaN
pois['lon'] = pd.to_numeric(pois['lon'], errors='coerce')
pois['lat'] = pd.to_numeric(pois['lat'], errors='coerce')

hexagons['left'] = pd.to_numeric(hexagons['left'], errors='coerce')
hexagons['top'] = pd.to_numeric(hexagons['top'], errors='coerce')
hexagons['right'] = pd.to_numeric(hexagons['right'], errors='coerce')
hexagons['bottom'] = pd.to_numeric(hexagons['bottom'], errors='coerce')

# Drop rows with invalid coordinates
pois = pois.dropna(subset=['lon', 'lat'])
hexagons = hexagons.dropna(subset=['left', 'top', 'right', 'bottom'])

# Calculate hexagon centroids
hexagons['centroid_lon'] = (hexagons['left'] + hexagons['right']) / 2
hexagons['centroid_lat'] = (hexagons['top'] + hexagons['bottom']) / 2

# Assign each POI to a hexagon
def assign_hexagon(poi_lon, poi_lat, hexagons):
    mask = (
        (hexagons['left'] <= poi_lon) & (poi_lon <= hexagons['right']) &
        (hexagons['bottom'] <= poi_lat) & (poi_lat <= hexagons['top'])
    )
    match = hexagons[mask]
    return match['id'].values[0] if len(match) > 0 else None

pois['hex_id'] = pois.apply(
    lambda row: assign_hexagon(row['lon'], row['lat'], hexagons), axis=1
)
# print(hexagons)
print("First 10 POIs with assigned hexagon IDs:")
print(pois[['name', 'lon', 'lat', 'hex_id']].head(10))

print("\nTotal POIs:", len(pois))
print("POIs assigned to hexagons:", pois['hex_id'].notna().sum())
print("POIs not assigned:", pois['hex_id'].isna().sum())

First 10 POIs with assigned hexagon IDs:
                      name        lon        lat hex_id
0        مسجد ملّا اسماعیل  54.363166  31.895873   None
1         مسجد شهید ساداتی  54.383829  31.829847   None
2               مسجد حظیره  54.371269  31.897907   None
3            حسینه گازرگاه  54.373771  31.893881   None
4  مسجد دانشگاه علوم پزشکی  54.340252  31.843144   None
5               مسجد ولایت  54.366102  31.826712   None
6                      NaN  54.351083  31.839957   None
7                      NaN  54.382941  31.899845   None
8                      NaN  54.383797  31.902155   None
9            مسجد جامع یزد  54.368511  31.901437   None

Total POIs: 1112
POIs assigned to hexagons: 0
POIs not assigned: 1112


cleaning data and prepare for next step (feature engineering)

Coordinate system mismatch.

POIs: Geographic coordinates (WGS84) — longitude ~54°, latitude ~32°
Hexagons: Projected coordinates (likely UTM or Web Mercator) — values in millions
You need to transform one dataset to match the other’s coordinate system. Here’s the solution using pyproj.

In [2]:
import pandas as pd
from pyproj import Transformer

# Load data
pois = pd.read_csv('Dataset.csv')
hexagons = pd.read_csv('Grid.csv')

# Convert to numeric
pois['lon'] = pd.to_numeric(pois['lon'], errors='coerce')
pois['lat'] = pd.to_numeric(pois['lat'], errors='coerce')
hexagons['left'] = pd.to_numeric(hexagons['left'], errors='coerce')
hexagons['top'] = pd.to_numeric(hexagons['top'], errors='coerce')
hexagons['right'] = pd.to_numeric(hexagons['right'], errors='coerce')
hexagons['bottom'] = pd.to_numeric(hexagons['bottom'], errors='coerce')

# Drop NaN
pois = pois.dropna(subset=['lon', 'lat'])
hexagons = hexagons.dropna(subset=['left', 'top', 'right', 'bottom'])

# Transform POI coordinates from WGS84 to Web Mercator (EPSG:3857)
# If your hexagons use a different projection (e.g., UTM Zone 40N for Iran), 
# replace 3857 with the appropriate EPSG code
transformer = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)
pois['x'], pois['y'] = transformer.transform(pois['lon'].values, pois['lat'].values)

# Calculate hexagon centroids
hexagons['center_x'] = (hexagons['left'] + hexagons['right']) / 2
hexagons['center_y'] = (hexagons['top'] + hexagons['bottom']) / 2

# Assign hexagons
def assign_hexagon(row):
    x, y = row['x'], row['y']
    mask = (
        (x >= hexagons['left']) & 
        (x <= hexagons['right']) & 
        (y >= hexagons['bottom']) & 
        (y <= hexagons['top'])
    )
    candidates = hexagons[mask]
    
    if len(candidates) == 0:
        return None
    
    distances = ((candidates['center_x'] - x)**2 + (candidates['center_y'] - y)**2)**0.5
    return candidates.loc[distances.idxmin(), 'id']

pois['hex_id'] = pois.apply(assign_hexagon, axis=1)

# Results
print("First 10 POIs with assigned hexagons:")
print(pois[['name', 'lon', 'lat', 'x', 'y', 'hex_id']].head(10))
print(f"\nTotal POIs: {len(pois)}")
print(f"Assigned: {pois['hex_id'].notna().sum()}")
print(f"Unassigned: {pois['hex_id'].isna().sum()}")

pois.to_csv('POIs_with_hexagons.csv', index=False)
print("\nSaved to POIs_with_hexagons.csv")


First 10 POIs with assigned hexagons:
                      name        lon        lat             x             y  \
0        مسجد ملّا اسماعیل  54.363166  31.895873  6.051680e+06  3.749650e+06   
1         مسجد شهید ساداتی  54.383829  31.829847  6.053980e+06  3.740996e+06   
2               مسجد حظیره  54.371269  31.897907  6.052582e+06  3.749917e+06   
3            حسینه گازرگاه  54.373771  31.893881  6.052861e+06  3.749389e+06   
4  مسجد دانشگاه علوم پزشکی  54.340252  31.843144  6.049129e+06  3.742738e+06   
5               مسجد ولایت  54.366102  31.826712  6.052007e+06  3.740585e+06   
6                      NaN  54.351083  31.839957  6.050335e+06  3.742321e+06   
7                      NaN  54.382941  31.899845  6.053881e+06  3.750171e+06   
8                      NaN  54.383797  31.902155  6.053977e+06  3.750474e+06   
9            مسجد جامع یزد  54.368511  31.901437  6.052275e+06  3.750380e+06   

    hex_id  
0  17205.0  
1  19432.0  
2  17983.0  
3  18297.0  
4  15055.0  
5  

Investigating the 7 unassigned POIs

In [3]:
print(pois[pois['hex_id'].isna()][['name', 'lon', 'lat']])

                        name        lon        lat
69   مجموعه ورزشی شهید نصیری  54.328391  31.858753
152                      NaN  54.313098  31.833249
230             ورزشگاه معلم  54.382191  31.920325
233   میدان میوه تره بار یزد  54.366374  31.792575
247                      NaN  54.328791  31.858544
356                      NaN  54.328777  31.858555
382            خانه ملک زاده  54.369065  31.904537


For each hexagon centroid, calculate distance to the nearest POI of each important type

In [6]:
# ============================================================
# CELL 3: Distance from each hexagon centroid to the nearest
#         POI of every fclass (POI type)
# ============================================================
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree          # efficient nearest-neighbour search

# ============================================================
# 1. Load the data you already prepared
# ============================================================
pois     = pd.read_csv('POIs_with_hexagons.csv')   # x, y, fclass
hexagons = pd.read_csv('Grid.csv')

# Make sure hexagon bounds are numeric and valid
for col in ['left', 'top', 'right', 'bottom']:
    hexagons[col] = pd.to_numeric(hexagons[col], errors='coerce')
hexagons = hexagons.dropna(subset=['left', 'top', 'right', 'bottom'])

# Hexagon centroids (in the SAME projected CRS – Web Mercator, EPSG:3857)
hexagons['center_x'] = (hexagons['left']   + hexagons['right'])  / 2
hexagons['center_y'] = (hexagons['top']    + hexagons['bottom']) / 2

# ============================================================
# 2. Discover all POI types (fclass) in the data
# ============================================================
unique_fclasses = pois['fclass'].dropna().unique()
print(f"Found {len(unique_fclasses)} POI types:")
for fc in unique_fclasses:
    cnt = (pois['fclass'] == fc).sum()
    print(f"   {fc}: {cnt} POIs")

# ============================================================
# 3. Build one cKDTree for each POI type
# ============================================================
trees = {}
for fc in unique_fclasses:
    subset = pois[pois['fclass'] == fc][['x', 'y']].dropna().values
    if len(subset) == 0:
        trees[fc] = None
        print(f"WARNING: No valid coordinates for '{fc}' – will return NaN")
    else:
        trees[fc] = cKDTree(subset)

# ============================================================
# 4. Query each hexagon centroid against every tree
# ============================================================
centroids = hexagons[['center_x', 'center_y']].values

# We'll store the distances in the hexagons DataFrame
for fc in unique_fclasses:
    tree = trees[fc]
    if tree is None:
        hexagons[f'dist_{fc}'] = np.nan
        continue

    # Query returns (distances, indices).  k=1  → nearest neighbour.
    dist, idx = tree.query(centroids, k=1)
    hexagons[f'dist_{fc}'] = dist          # distance in meters (Web Mercator)

# ============================================================
# 5. (Optional) Add a column for the absolute nearest POI type & distance
# ============================================================
dist_cols = [f'dist_{fc}' for fc in unique_fclasses]
hexagons['min_dist_overall'] = hexagons[dist_cols].min(axis=1)
hexagons['nearest_type']     = hexagons[dist_cols].idxmin(axis=1).str.replace('dist_', '')

# ============================================================
# 6. Preview and save
# ============================================================
print("\n--- Sample of the result ---")
display_cols = ['id', 'center_x', 'center_y', 'min_dist_overall', 'nearest_type'] + dist_cols
print(hexagons[display_cols].head(10))

hexagons.to_csv('Hexagons_with_nearest_POI_distances.csv', index=False)
print("\n✅ Saved to Hexagons_with_nearest_POI_distances.csv")


Found 82 POI types:
   muslim: 24 POIs
   muslim_shia: 30 POIs
   jewish: 10 POIs
   theme_park: 2 POIs
   graveyard: 19 POIs
   sports_centre: 12 POIs
   park: 135 POIs
   attraction: 45 POIs
   university: 27 POIs
   school: 64 POIs
   mall: 8 POIs
   hospital: 15 POIs
   bank: 54 POIs
   department_store: 4 POIs
   clinic: 22 POIs
   hotel: 68 POIs
   pitch: 9 POIs
   restaurant: 63 POIs
   swimming_pool: 5 POIs
   guesthouse: 13 POIs
   college: 9 POIs
   library: 7 POIs
   cinema: 4 POIs
   shelter: 1 POIs
   convenience: 8 POIs
   fire_station: 4 POIs
   stadium: 2 POIs
   car_dealership: 12 POIs
   monument: 2 POIs
   market_place: 5 POIs
   police: 11 POIs
   museum: 8 POIs
   prison: 2 POIs
   courthouse: 2 POIs
   town_hall: 3 POIs
   castle: 2 POIs
   cafe: 39 POIs
   supermarket: 41 POIs
   fast_food: 35 POIs
   hairdresser: 11 POIs
   optician: 10 POIs
   jeweller: 11 POIs
   pharmacy: 20 POIs
   bookshop: 3 POIs
   computer_shop: 8 POIs
   mobile_phone_shop: 13 POIs
   at

In [7]:
# ============================================================
# CELL 4: Count POIs within a fixed radius (e.g., 500m)
#         from each hexagon centroid, per fclass
# ============================================================
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# ============================================================
# 1. Load data
# ============================================================
pois     = pd.read_csv('POIs_with_hexagons.csv')   # x, y, fclass
hexagons = pd.read_csv('Grid.csv')

# Ensure hexagon bounds are numeric
for col in ['left', 'top', 'right', 'bottom']:
    hexagons[col] = pd.to_numeric(hexagons[col], errors='coerce')
hexagons = hexagons.dropna(subset=['left', 'top', 'right', 'bottom'])

# Hexagon centroids (same projection as POIs — Web Mercator, EPSG:3857)
hexagons['center_x'] = (hexagons['left']   + hexagons['right'])  / 2
hexagons['center_y'] = (hexagons['top']    + hexagons['bottom']) / 2

# ============================================================
# 2. Set your radius (in meters — Web Mercator units)
# ============================================================
RADIUS = 500   # ← change this to any value you need

# ============================================================
# 3. Discover all POI types
# ============================================================
unique_fclasses = pois['fclass'].dropna().unique()
print(f"Counting POIs within {RADIUS}m for {len(unique_fclasses)} types:")
for fc in unique_fclasses:
    print(f"   {fc}: {(pois['fclass'] == fc).sum()} total POIs")

# ============================================================
# 4. Build one cKDTree per POI type
# ============================================================
trees = {}
for fc in unique_fclasses:
    subset = pois[pois['fclass'] == fc][['x', 'y']].dropna().values
    if len(subset) == 0:
        trees[fc] = None
    else:
        trees[fc] = cKDTree(subset)

# ============================================================
# 5. Count POIs within radius for each hexagon centroid
# ============================================================
centroids = hexagons[['center_x', 'center_y']].values

for fc in unique_fclasses:
    tree = trees[fc]
    if tree is None:
        hexagons[f'count_{fc}'] = 0
        continue

    # query_ball_point returns indices of all POIs within RADIUS
    idx_list = tree.query_ball_point(centroids, r=RADIUS)

    # Count how many POIs in each hexagon's list
    hexagons[f'count_{fc}'] = [len(lst) for lst in idx_list]

# ============================================================
# 6. Add a total count across all types
# ============================================================
count_cols = [f'count_{fc}' for fc in unique_fclasses]
hexagons['count_total'] = hexagons[count_cols].sum(axis=1)

# ============================================================
# 7. Preview and save
# ============================================================
display_cols = ['id', 'center_x', 'center_y', 'count_total'] + count_cols
print(f"\n--- Hexagons with POI counts within {RADIUS}m ---")
print(hexagons[display_cols].head(10))

# Quick summary
print(f"\nHexagons with at least one POI within {RADIUS}m: "
      f"{(hexagons['count_total'] > 0).sum()} / {len(hexagons)}")

hexagons.to_csv('Hexagons_with_POI_counts.csv', index=False)
print("\n✅ Saved to Hexagons_with_POI_counts.csv")


Counting POIs within 500m for 82 types:
   muslim: 24 total POIs
   muslim_shia: 30 total POIs
   jewish: 10 total POIs
   theme_park: 2 total POIs
   graveyard: 19 total POIs
   sports_centre: 12 total POIs
   park: 135 total POIs
   attraction: 45 total POIs
   university: 27 total POIs
   school: 64 total POIs
   mall: 8 total POIs
   hospital: 15 total POIs
   bank: 54 total POIs
   department_store: 4 total POIs
   clinic: 22 total POIs
   hotel: 68 total POIs
   pitch: 9 total POIs
   restaurant: 63 total POIs
   swimming_pool: 5 total POIs
   guesthouse: 13 total POIs
   college: 9 total POIs
   library: 7 total POIs
   cinema: 4 total POIs
   shelter: 1 total POIs
   convenience: 8 total POIs
   fire_station: 4 total POIs
   stadium: 2 total POIs
   car_dealership: 12 total POIs
   monument: 2 total POIs
   market_place: 5 total POIs
   police: 11 total POIs
   museum: 8 total POIs
   prison: 2 total POIs
   courthouse: 2 total POIs
   town_hall: 3 total POIs
   castle: 2 total